# CURB - Unified Hotspot Intelligence (capstone)
### notebook 05 - Problem Statement 1 (Parking-Induced Congestion)

Every CURB signal, merged into one enriched record per hotspot:

**priority / impact + root cause + fix + persistence** (prioritizer)
**+ trend** (Emerging / Stable / Cooling)
**+ recidivism** (Habitual / Mixed / Transient - same vehicles returning?)

This produces `curb_intelligence.json` - a **superset** of `curb_priorities.json`
(same fields + `trend`, `recidivism`), so it's an additive, drop-in upgrade for the
frontend. It also surfaces the **high-conviction watchlist**: spots that are
high-impact AND emerging AND habitual - the worst, getting worse, with the same
vehicles returning.

Provided dataset only. (Runs the three analyses, so expect ~30s.)

In [ ]:
import os, sys, json
from pathlib import Path
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "config.py").exists() and (REPO_ROOT.parent / "config.py").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT)); os.chdir(REPO_ROOT)
import config, pandas as pd, matplotlib.pyplot as plt
from src.intelligence import run
DATA_PATH = config.DATA_PATH
pd.set_option("display.max_colwidth", 30)
print("data:", DATA_PATH, "exists:", os.path.exists(DATA_PATH))

## 1 - Build the unified layer

In [ ]:
m = run(DATA_PATH)
print(f"{len(m):,} hotspots, each carrying every signal")

## 2 - The unified profile (every signal in one view)

In [ ]:
m.head(15)[["rank","name","congestion_impact_index","root_cause",
            "persistence_tier","trend","recidivism"]]

## 3 - The high-conviction watchlist
The narrowing: of all hotspots, which are **top-100 impact AND Emerging AND Habitual**?
Worst, getting worse, same vehicles returning - start here.

In [ ]:
hc = m[m.high_conviction][["rank","name","congestion_impact_index","root_cause",
                            "trend","recidivism","repeat_share"]]
print(f"High-conviction (all three criteria): {len(hc)} of {len(m):,} hotspots")
hc

### Broader "elevated" watchlist
The strict filter is deliberately tight. A usable operational tier: **top-100 impact
that are Emerging OR Habitual** - still sharply narrowed, but a workable list.

In [ ]:
elevated = m[(m["rank"] <= 100) & ((m.trend == "Emerging") | (m.recidivism == "Habitual"))]
elevated[["rank","name","congestion_impact_index","root_cause","trend","recidivism"]].head(15)

## 4 - Structure: how the signals relate

In [ ]:
print("Root cause x Trend (which problems are escalating?)")
print(pd.crosstab(m.root_cause, m.trend))
print("\nPersistence x Recidivism (do chronic spots have returning vehicles?)")
print(pd.crosstab(m.persistence_tier, m.recidivism))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ct = pd.crosstab(m.root_cause, m.trend)
ct = ct[[c for c in ["Emerging","Stable","Cooling"] if c in ct.columns]]
ct.plot.barh(ax=ax, stacked=True, color={"Emerging":"#E5564B","Stable":"#7C8DA6","Cooling":"#4C9BE6"})
ax.set_title("Root cause by trend"); ax.set_xlabel("hotspots"); plt.tight_layout(); plt.show()

## 5 - File written

In [ ]:
print("wrote outputs/curb_intelligence.json -", os.path.getsize("outputs/curb_intelligence.json"), "bytes")
j = json.load(open("outputs/curb_intelligence.json"))
print("meta:", json.dumps(j["meta"], indent=2))
print("\nenriched hotspot record (note trend + recidivism added):")
print(json.dumps(j["hotspots"][0], indent=2))

## Methodology note
- This merges signals computed in notebooks 02 (priority/impact + root cause +
  persistence), 04 (trend), and 03 (location recidivism). Each is provided-data only.
- `curb_intelligence.json` is a superset of `curb_priorities.json`; existing fields are
  unchanged, so the frontend can adopt it without breaking (new badges: trend, recidivism).
- `congestion_impact_index` remains a normalised index, not measured traffic.
- "High-conviction" is a transparent AND of three flags; the broader tier is an OR. Both
  are explainable to the panel.